In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
try:
    df = pd.read_csv("data/synth_data_for_training.csv")
except FileNotFoundError:
    print("Error: 'data/synth_data_for_training.csv' not found.")
    df = pd.DataFrame() 

if not df.empty:
    print("=== EXACT VALIDITY STATISTICS FOR REPORT ===\n")
    sns.set_style("whitegrid")
    
    # ==========================================
    # CHART 1: AGE BIAS (Granular)
    # ==========================================
    if 'persoon_leeftijd_bij_onderzoek' in df.columns:
        # Create bins
        bins = [0, 25, 35, 45, 55, 120]
        labels = ['< 25', '25-34', '35-44', '45-54', '55+']
        df['age_group_detailed'] = pd.cut(df['persoon_leeftijd_bij_onderzoek'], bins=bins, labels=labels)
        
        # Calculate stats
        age_stats = df.groupby('age_group_detailed', observed=False)['checked'].mean() * 100
        
        print(f"--- 1. Age Bias (Detailed Breakdown) ---")
        for group in labels:
            print(f"Flag Rate ({group}): {age_stats[group]:.2f}%")
        print(f"-> Disparity: Under-25s are flagged {age_stats['< 25'] / age_stats['55+']:.2f}x more often than over-55s.\n")
        
        # Generate Figure 1
        plt.figure(figsize=(8, 5))
        sns.barplot(x=age_stats.index, y=age_stats.values, palette="Blues_d", hue=age_stats.index, legend=False)
        plt.title('Appendix Figure 1: Flag Rate by Age Group')
        plt.ylabel('Flag Rate (%)')
        plt.xlabel('Age Group')
        plt.ylim(0, age_stats.max() * 1.2)
        
        # Add labels
        for i, v in enumerate(age_stats.values):
             plt.text(i, v + 0.5, f"{v:.1f}%", ha='center', fontweight='bold')
             
        plt.tight_layout()
        plt.savefig('appendix_age_bias.png')
        plt.close() # Close plot to free memory

    # ==========================================
    # CHART 2: STABILITY BIAS (Recent Movers)
    # ==========================================
    if 'adres_dagen_op_adres' in df.columns:
        df['new_resident'] = df['adres_dagen_op_adres'] < 365
        df['long_term'] = df['adres_dagen_op_adres'] > (365 * 5)
        
        new_rate = df[df['new_resident'] == True]['checked'].mean() * 100
        long_rate = df[df['long_term'] == True]['checked'].mean() * 100
        
        print(f"--- 2. Stability Bias (Moving House) ---")
        print(f"Flag Rate (New Residents < 1yr): {new_rate:.2f}%")
        print(f"Flag Rate (Long Term > 5yrs):    {long_rate:.2f}%")
        print(f"-> Disparity: Recent movers are flagged {new_rate / long_rate:.2f}x more often.\n")
        
        # Generate Figure 2
        plt.figure(figsize=(6, 5))
        x_data = ['New (<1 yr)', 'Long Term (>5 yrs)']
        y_data = [new_rate, long_rate]
        
        sns.barplot(x=x_data, y=y_data, palette="Greens_d", hue=x_data, legend=False)
        plt.title('Appendix Figure 2: Flag Rate by Residence Duration')
        plt.ylabel('Flag Rate (%)')
        plt.ylim(0, max(new_rate, long_rate) * 1.2)
        
        for i, v in enumerate(y_data):
             plt.text(i, v + 0.5, f"{v:.1f}%", ha='center', fontweight='bold')
             
        plt.tight_layout()
        plt.savefig('appendix_stability_bias.png')
        plt.close()

    # ==========================================
    # CHART 3: POLICY BIAS (Kostendeler)
    # ==========================================
    if 'relatie_overig_kostendeler' in df.columns:
        kd_stats = df.groupby('relatie_overig_kostendeler')['checked'].agg(['mean', 'count'])
        alone_rate = kd_stats.loc[0, 'mean'] * 100
        sharer_rate = kd_stats.loc[1, 'mean'] * 100
        
        print(f"--- 3. Policy Bias (Kostendeler) ---")
        print(f"Flag Rate (Living Alone): {alone_rate:.2f}%")
        print(f"Flag Rate (Cost Sharer):  {sharer_rate:.2f}%")
        print(f"-> Disparity: Housemates are flagged {sharer_rate / alone_rate:.2f}x more often.\n")
        
        # Generate Figure 3
        plt.figure(figsize=(6, 5))
        x_data = ['Living Alone', 'Cost Sharer']
        y_data = [alone_rate, sharer_rate]
        
        sns.barplot(x=x_data, y=y_data, palette="Reds_d", hue=x_data, legend=False)
        plt.title('Appendix Figure 3: Flag Rate by Cost Sharer Status')
        plt.ylabel('Flag Rate (%)')
        plt.ylim(0, max(alone_rate, sharer_rate) * 1.2)
        
        for i, v in enumerate(y_data):
             plt.text(i, v + 0.5, f"{v:.1f}%", ha='center', fontweight='bold')
             
        plt.tight_layout()
        plt.savefig('appendix_policy_bias.png')
        plt.close()

    print("Success: Three separate charts saved to folder.")

=== EXACT VALIDITY STATISTICS FOR REPORT ===

--- 1. Age Bias (Detailed Breakdown) ---
Flag Rate (< 25): 35.52%
Flag Rate (25-34): 25.02%
Flag Rate (35-44): 14.79%
Flag Rate (45-54): 6.20%
Flag Rate (55+): 5.28%
-> Disparity: Under-25s are flagged 6.73x more often than over-55s.

--- 2. Stability Bias (Moving House) ---
Flag Rate (New Residents < 1yr): 25.13%
Flag Rate (Long Term > 5yrs):    9.28%
-> Disparity: Recent movers are flagged 2.71x more often.

--- 3. Policy Bias (Kostendeler) ---
Flag Rate (Living Alone): 7.50%
Flag Rate (Cost Sharer):  17.56%
-> Disparity: Housemates are flagged 2.34x more often.

Success: Three separate charts saved to folder.
